# Blanket MARL visual evaluation

Run one deterministic recurrent-policy trajectory, or a seeded random-action trajectory when no trained checkpoint exists. After every physical control step, the notebook displays the action norm and reward histories followed by the three standardized wall heat-flux observation channels. This launches one NekRS/SmartSim environment only when the code cell is executed.

In [ ]:
import json
import os
import socket
import sys
from pathlib import Path

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import clear_output, display
from sb3_contrib import RecurrentPPO

working_directory = Path.cwd().resolve()
root_candidates = (working_directory, *working_directory.parents)
PROJECT_ROOT = next((path for path in root_candidates if (path / 'pyproject.toml').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Run this notebook from the MagnetoConvectiveControl repository')
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from magnetoconvective_control.envs.smartsim_env import (
    SB3MARLSmartSimVecEnv, SmartSimVecEnv, WALL_CHANNELS,
)

MODEL_NAME = os.environ.get('RL_MODEL_NAME', 'first_wall_ppo')
MODEL_PATH = Path(os.environ.get(
    'MCC_CHECKPOINT', PROJECT_ROOT / 'runs' / MODEL_NAME / 'checkpoints' / 'step_000000275000.zip'
)).expanduser().resolve()
CASE_DIR = PROJECT_ROOT / 'cases' / 'toroidal_gr1e9_ha1750'
SIM_EXECUTABLE = Path(os.environ.get('RL_SIM_EXECUTABLE', CASE_DIR / 'run_case.sh')).resolve()
SNAPSHOTS_DIR = Path(os.environ.get('RL_SNAPSHOTS_DIR', PROJECT_ROOT / 'data' / 'snapshots')).resolve()
LAUNCHER = os.environ.get('RL_LAUNCHER', 'local')
DATABASE_INTERFACE = os.environ.get('SMARTSIM_INTERFACE', 'lo')
database_port_override = os.environ.get('SMARTSIM_PORT')
if database_port_override is None:
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as port_probe:
        port_probe.bind(('127.0.0.1', 0))
        DATABASE_PORT = int(port_probe.getsockname()[1])
else:
    DATABASE_PORT = int(database_port_override)
print(f'SmartSim database port: {DATABASE_PORT}')
EPISODE_STEPS = 487 #int(os.environ.get('RL_TEST_STEPS', '1000'))
WALL_NPERP = int(os.environ.get('RL_WALL_NPERP', '128'))
WALL_NZ = int(os.environ.get('RL_WALL_NZ', '160'))
N_AGENTS = 10

mapping_path = MODEL_PATH.parents[1] / 'action_mapping.json'
if MODEL_PATH.is_file() and not mapping_path.is_file():
    raise FileNotFoundError(f'Action mapping not found: {mapping_path}')
saved_mapping = json.loads(mapping_path.read_text()) if MODEL_PATH.is_file() else {'baseline': 0.0, 'scale': 1.0}
ACTION_BASELINE = float(saved_mapping['baseline'])
ACTION_SCALE = float(saved_mapping['scale'])
RANDOM_SEED = int(os.environ.get('RL_TEST_SEED', '5489'))
random_generator = np.random.default_rng(RANDOM_SEED)

# Recreate the native worker environment when Jupyter was started without smoke_test.sh.
CONDA_PREFIX = Path(os.environ.get('CONDA_PREFIX', sys.prefix)).resolve()
NEKRS_HOME = Path(os.environ.get('NEKRS_HOME', Path.home() / '.local' / 'nekrs-mhd')).resolve()
SMARTREDIS_LIBRARY = CONDA_PREFIX / 'lib' / 'libsmartredis.so'
if not (NEKRS_HOME / 'bin' / 'nrsmpi').is_file():
    raise FileNotFoundError(f'nrsmpi was not found under NEKRS_HOME={NEKRS_HOME}')
if not SMARTREDIS_LIBRARY.is_file():
    raise FileNotFoundError(f'SmartRedis library not found: {SMARTREDIS_LIBRARY}')
os.environ['NEKRS_HOME'] = str(NEKRS_HOME)
os.environ['NEKRS_UDF_INCLUDES'] = str(CONDA_PREFIX / 'include')
os.environ['NEKRS_UDF_LIBS'] = str(SMARTREDIS_LIBRARY)
os.environ.setdefault('BLANKET_CMAKE', '/usr/bin/cmake')
os.environ.setdefault('OMP_NUM_THREADS', '1')

observation_space = gym.spaces.Dict({
    'heat_flux': gym.spaces.Box(
        low=-np.inf, high=np.inf, shape=(3, WALL_NPERP, WALL_NZ), dtype=np.float32
    ),
    'previous_actions': gym.spaces.Box(
        low=-1.0, high=1.0, shape=(N_AGENTS,), dtype=np.float32
    ),
})
action_space = gym.spaces.Box(low=-0.25, high=0.25, shape=(1,), dtype=np.float32)

core = None
env = None
try:
    core = SmartSimVecEnv(
        n_envs=1, simulations_per_gpu=1, sim_executable=str(SIM_EXECUTABLE),
        wall_shape=(WALL_NPERP, WALL_NZ), act_dim=N_AGENTS, launcher=LAUNCHER,
        gpu_gres=os.environ.get('RL_GPU_GRES', 'gpu:1'),
        cpus_per_simulation=int(os.environ.get('RL_CPUS_PER_SIMULATION', '1')),
        max_episode_steps=None, snapshots_dir=str(SNAPSHOTS_DIR), startup_timeout=500.0,
        database_port=DATABASE_PORT, database_interface=DATABASE_INTERFACE,
        action_baseline=ACTION_BASELINE, action_scale=ACTION_SCALE,
        first_wall_heat_flux_baseline=float(os.environ.get('RL_FIRST_WALL_HEAT_FLUX_BASELINE', '0.0166913')),
        reward_scale=float(os.environ.get('RL_REWARD_SCALE', '1.0')),
        ewma_timescale=float(os.environ.get('RL_EWMA_TIMESCALE', '10.0')),
    )
    env = SB3MARLSmartSimVecEnv(core, observation_space, action_space)
    model = RecurrentPPO.load(str(MODEL_PATH), env=env, device='cpu') if MODEL_PATH.is_file() else None
    observations = env.reset()
    lstm_states = None
    episode_starts = np.ones(env.num_envs, dtype=bool)
    action_history = []
    reward_history = []
    first_wall_flux_history = []

    if model is None:
        print(f'No model found at {MODEL_PATH}; using seeded random actuation (seed={RANDOM_SEED}).')
    else:
        print(f'Loaded {MODEL_PATH}')
    print(f'Snapshot: {core._episode_snapshots[0]}')
    for step in range(1, EPISODE_STEPS + 1):
        if 5 <= step < 15:
            observations['previous_actions'][...] = 0.0
        if model is None:
            policy_actions = -0.25 * np.ones((env.num_envs, 1), dtype=np.float32) #* random_generator.uniform(-0.25, 0.25)
        else:
            policy_actions, lstm_states = model.predict(
                observations, state=lstm_states, episode_start=episode_starts, deterministic=True
            )
        observations, rewards, dones, infos = env.step(policy_actions)
        physical_action = np.asarray(infos[0]['physical_action'], dtype=np.float64)
        action_history.append(physical_action.copy())
        reward_history.append(float(rewards[0]))
        first_wall_flux_history.append(float(infos[0]['first_wall_heat_flux']))
        fields = np.asarray(core._last_observations[0])

        fig, axes = plt.subplots(5, 1, figsize=(7, 10), constrained_layout=True)
        steps = np.arange(1, step + 1)
        action_values = np.asarray(action_history)
        axes[0].plot(steps, action_values, linewidth=0.7)
        axes[0].set(title='Physical electrode actions', ylabel='action', xlabel='control step')
        axes[0].grid(True, alpha=0.3)
        reward_axis = axes[1]
        flux_axis = reward_axis.twinx()
        reward_axis.plot(steps, reward_history, color='tab:orange', label='reward')
        flux_axis.plot(steps, first_wall_flux_history, color='tab:red', alpha=0.8,
                       label='reactor-facing wall flux')
        reward_axis.axhline(0.0, color='black', linewidth=0.8)
        reward_axis.set(title='Reward and reactor-facing wall heat flux',
                        ylabel='reward', xlabel='control step')
        reward_axis.tick_params(axis='y', labelcolor='tab:orange')
        flux_axis.set_ylabel('integrated heat flux', color='tab:red')
        flux_axis.tick_params(axis='y', labelcolor='tab:red')
        reward_axis.grid(True, alpha=0.3)
        field_limit = max(float(np.max(np.abs(fields))), 1e-8)
        for channel, (axis, wall_name) in enumerate(zip(axes[2:], WALL_CHANNELS)):
            image = axis.imshow(
                fields[channel], origin='lower', aspect='auto', cmap='coolwarm',
                vmin=-field_limit, vmax=field_limit, interpolation='nearest',
            )
            axis.set(title=f'{wall_name} standardized outward heat flux — step {step}',
                     ylabel='transverse index', xlabel='z index')
            fig.colorbar(image, ax=axis, label='standardized heat flux')
        clear_output(wait=True)
        fig.suptitle(f'Control step {step} (cumulative reward: {sum(reward_history):.2f})', fontsize=16)
        display(fig)
        plt.close(fig)

        episode_starts = np.asarray(dones, dtype=bool)
        if np.any(dones):
            print(f'Environment ended after {step} control steps.')
            break
finally:
    if env is not None:
        env.close()
    elif core is not None:
        core.close()
